In [14]:
# Vulnerability CLassifier (NN): i feel like I would use a normal ANN
# GOAL: Given vulnerability attributes and description predict a CVSS score (float value from 0-10)
# INPUT:
# OUTPUT: value between 0-10
# https://www.kaggle.com/code/cloudnineforreal/cvss-prediction

In [15]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

## Upload/Understand Data

In [16]:
cve_data = pd.read_csv("../data/cve.csv")  # data from kaggle
cve_data.head()

,Unnamed: 0,mod_date,pub_date,cvss,cwe_code,cwe_name,summary,access_authentication,access_complexity,access_vector,impact_availability,impact_confidentiality,impact_integrity
0,CVE-2019-16548,2019-11-21 15:15:00,2019-11-21 15:15:00,6.8,352,Cross-Site Request Forgery (CSRF),A cross-site request forgery vulnerability in ...,NaN,NaN,NaN,NaN,NaN,NaN
1,CVE-2019-16547,2019-11-21 15:15:00,2019-11-21 15:15:00,4.0,732,Incorrect Permission Assignment for Critical ...,Missing permission checks in various API endpo...,NaN,NaN,NaN,NaN,NaN,NaN
2,CVE-2019-16546,2019-11-21 15:15:00,2019-11-21 15:15:00,4.3,639,Authorization Bypass Through User-Controlled Key,Jenkins Google Compute Engine Plugin 4.1.1 and...,NaN,NaN,NaN,NaN,NaN,NaN
3,CVE-2013-2092,2019-11-20 21:22:00,2019-11-20 21:15:00,4.3,79,Improper Neutralization of Input During Web P...,Cross-site Scripting (XSS) in Dolibarr ERP/CRM...,NaN,NaN,NaN,NaN,NaN,NaN
4,CVE-2013-2091,2019-11-20 20:15:00,2019-11-20 20:15:00,7.5,89,Improper Neutralization of Special Elements u...,SQL injection vulnerability in Dolibarr ERP/CR...,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# drop columns
drop_col = ["Unnamed: 0", "mod_date", "pub_date"]
cve_data.drop(columns=drop_col, inplace=True)

# define X and y
y = cve_data["cvss"]
X = cve_data.drop(columns=["cvss"])

In [18]:
# fill na categorical columns as "UNKNOWN"
catgy_cols = ["access_authentication", "access_complexity", "access_vector", "impact_availability", "impact_confidentiality", "impact_integrity"]
# cve_data[catgy_cols] = cve_data[catgy_cols].fillna("UNKNOWN")
for col in catgy_cols:
    if col in X.columns:
        X[col] = X[col].fillna("UNKNOWN").str.upper()

# one hot encode categorical columns
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).set_output(transform="pandas")
ohe.fit(X[catgy_cols])
catgy_encode = ohe.transform(X[catgy_cols])

# combine data
X_processed = pd.concat([X.drop(columns=catgy_cols), catgy_encode], axis=1)
# cve_data.drop(columns=catgy_cols, inplace=True)

In [19]:
cve_data.isna().sum().sum()

5304

In [20]:
cve_data["summary"]

0        A cross-site request forgery vulnerability in ...
1        Missing permission checks in various API endpo...
2        Jenkins Google Compute Engine Plugin 4.1.1 and...
3        Cross-site Scripting (XSS) in Dolibarr ERP/CRM...
4        SQL injection vulnerability in Dolibarr ERP/CR...
                               ...                        
89655    ** REJECT **  DO NOT USE THIS CANDIDATE NUMBER...
89656    ** REJECT **  DO NOT USE THIS CANDIDATE NUMBER...
89657    ** REJECT **  DO NOT USE THIS CANDIDATE NUMBER...
89658    ** REJECT **  DO NOT USE THIS CANDIDATE NUMBER...
89659    ** REJECT **  DO NOT USE THIS CANDIDATE NUMBER...
Name: summary, Length: 89660, dtype: object

In [21]:
# vectorize summary field (SBERT)
sbert_model = SentenceTransformer("all-MiniLM-L6-v2") 
embeddings = sbert_model.encode(X_processed["summary"])
embeddings_df = pd.DataFrame(
    embeddings,
    columns=[f"SBERT_summary_{i}" for i in range(embeddings.shape[1])]
)

X_processed = pd.concat([X_processed.drop(columns=["summary"]), embeddings_df], axis=1)

# tfidf_summary = TfidfVectorizer(max_features=500, stop_words="english")
# summary_feat = tfidf_summary.fit_transform(cve_data["summary"])
# # print(summary_feat[6])
# summary_feat_df = pd.DataFrame(
#     summary_feat.toarray(),
#     columns=[f"tfidf_summary_{i}" for i in range(summary_feat.shape[1])]
# )

# # combine data
# merged_cve_data = pd.concat([cve_data.drop(columns=["summary"]), summary_feat_df], axis=1)

# vectorize cve name field
tfidf_name = TfidfVectorizer(max_features=50, stop_words="english")
cwe_name_feat = tfidf_name.fit_transform(X_processed["cwe_name"])
name_feat_df = pd.DataFrame(
    cwe_name_feat.toarray(),
    columns=[f"tfidf_name_{i}" for i in range(cwe_name_feat.shape[1])]
)

# combine data
X_processed = pd.concat([X_processed.drop(columns=["cwe_name"]), name_feat_df], axis=1)
X_processed.head()

,cwe_code,access_authentication_MULTIPLE,access_authentication_NONE,access_authentication_SINGLE,access_authentication_UNKNOWN,access_complexity_HIGH,access_complexity_LOW,access_complexity_MEDIUM,access_complexity_UNKNOWN,access_vector_ADJACENT_NETWORK,...,tfidf_name_40,tfidf_name_41,tfidf_name_42,tfidf_name_43,tfidf_name_44,tfidf_name_45,tfidf_name_46,tfidf_name_47,tfidf_name_48,tfidf_name_49
0,352,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.000000,0.315817,0.00000,0.000000,0.0,0.0,0.000000,0.0,0.000000
1,732,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.000000,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.0,0.000000
2,639,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.000000,0.000000,0.00000,0.000000,0.0,0.0,0.000000,0.0,0.000000
3,79,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.369922,0.345435,0.00000,0.000000,0.0,0.0,0.000000,0.0,0.369922
4,89,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.000000,0.000000,0.31294,0.673061,0.0,0.0,0.312976,0.0,0.000000


In [23]:
# save the feature schema
import json
feature_names = X_processed.columns.tolist()
with open("../model/feature_schema.json", "w") as f:
    json.dump({"features": feature_names}, f, indent=2)

In [28]:
X_processed.to_csv("processed_cve_data.csv")

In [24]:
# save encoders
import joblib

joblib.dump(ohe, "../model/ohe_encoder.pkl")
joblib.dump(tfidf_name, "../model/tfodf_encoder.pkl")

['../model/tfodf_encoder.pkl']

# Merged Data Run

In [ ]:
# merged_cve_data = pd.read_csv("C:\\Users\\bride\\OneDrive\\Desktop\\merged_cve.csv")  # processed data

In [ ]:
# merged_cve_data.isna().sum().sum()

0

In [ ]:
# # split train/test
# input_cols = merged_cve_data.loc[:, merged_cve_data.columns != "cvss"].columns
# # input_cols
# X = merged_cve_data[input_cols]
# y = merged_cve_data["cvss"]

# # drop object columns
# X = X.select_dtypes(exclude="object")

In [ ]:
# input_cols

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.3, random_state=42)

## Model Definition

In [26]:
# UPDATED MODEL DEFINITION
dtrain = xgb.DMatrix(X_train, label=y_train)
params = {
    "objective": "reg:squarederror",
    "max_depth": 6,
    "learning_rate": 0.1,
    "n_estimators": 100
}

# Create DMatrix objects for training and testing sets
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

In [27]:
# vuln_model = xgb.train(params, dtrain, num_boost_round=100)
import numpy as np

# Train the XGBoost model
num_rounds = 100
model = xgb.train(params, dtrain, num_rounds, evals=[(dtest, 'test')], early_stopping_rounds=10)

# Make predictions on the test set
y_pred = model.predict(dtest)

# Evaluate the model's performance
rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))
print(f"Test RMSE: {rmse:.2f}")

c:\Users\bride\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py:158: UserWarning: [10:44:40] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "n_estimators" } are not used.

  warnings.warn(smsg, UserWarning)


[0]	test-rmse:1.80111
[1]	test-rmse:1.62982
[2]	test-rmse:1.47665
[3]	test-rmse:1.33898
[4]	test-rmse:1.21567
[5]	test-rmse:1.10364
[6]	test-rmse:1.00383
[7]	test-rmse:0.91322
[8]	test-rmse:0.83425
[9]	test-rmse:0.76324
[10]	test-rmse:0.69735
[11]	test-rmse:0.63950
[12]	test-rmse:0.58761
[13]	test-rmse:0.54213
[14]	test-rmse:0.49985
[15]	test-rmse:0.46379
[16]	test-rmse:0.42856
[17]	test-rmse:0.39911
[18]	test-rmse:0.37390
[19]	test-rmse:0.34939
[20]	test-rmse:0.32624
[21]	test-rmse:0.30654
[22]	test-rmse:0.29105
[23]	test-rmse:0.27585
[24]	test-rmse:0.26295
[25]	test-rmse:0.25203
[26]	test-rmse:0.24295
[27]	test-rmse:0.23440
[28]	test-rmse:0.22709
[29]	test-rmse:0.22090
[30]	test-rmse:0.21560
[31]	test-rmse:0.21044
[32]	test-rmse:0.20627
[33]	test-rmse:0.20270
[34]	test-rmse:0.19978
[35]	test-rmse:0.19722
[36]	test-rmse:0.19500
[37]	test-rmse:0.19226
[38]	test-rmse:0.19064
[39]	test-rmse:0.18947
[40]	test-rmse:0.18815
[41]	test-rmse:0.18712
[42]	test-rmse:0.18565
[43]	test-rmse:0.1846

In [ ]:
vuln_model.

In [ ]:
# vuln_regr = XGBRegressor()
# model = vuln_regr.fit(X_train, y_train)

In [ ]:
# save model
# vuln_regr.save_model("../model/xgb_regressor.json")

In [ ]:
# model metrics
from sklearn.metrics import mean_squared_error, r2_score

# oob = rfr.oob_score_
# print(f"Out of Bag Score: {oob}")

predict = vuln_regr.predict(X_test)
mse = mean_squared_error(y_test, predict)
print(f"MSE: {mse}")

r2 = r2_score(y_test, predict)
print(f"R2 Value: {r2}")


MSE: 0.02625544087595387
R2 Value: 0.9933818153679375
